# SCL CP8D 0.8 µm PMOS Characterization

This notebook analyses PMOS transfer curves at fixed `VSD` and output curves at fixed `VSG`. It calculates numerical `gm` and `gm/ID` estimates from the transfer curves.

**Sign convention:** use magnitudes `VSG`, `VSD`, and `|ID|` for PMOS plots. The raw `wrdata` columns from the current netlists were inspected: transfer/output voltage vector is in column 2 (index 1), and current is in the last column.

**Model caveat:** Ngspice 42 reported ignored HSPICE-specific parameters for the SCL model. Treat extracted values as preliminary until model compatibility is validated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set this to the directory containing the raw Ngspice .dat files.
DATA_DIR = Path("../data/raw")
OUT_TABLES = Path("../results/tables")
OUT_FIGURES = Path("../results/figures")
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

def load_wrdata(path, x_col=1, current_col=-1):
    """Load Ngspice wrdata output; return sorted x values and |current| in A."""
    data = np.loadtxt(path)
    x = data[:, x_col]
    current = np.abs(data[:, current_col])
    order = np.argsort(x)
    return x[order], current[order]

In [ ]:
transfer_files = {
    0.5: "scl_pmos_transfer_vsd_0p5.dat",
    1.0: "scl_pmos_transfer_vsd_1.dat",
    2.0: "scl_pmos_transfer_vsd_2.dat",
    3.0: "scl_pmos_transfer_vsd_3.dat",
    4.0: "scl_pmos_transfer_vsd_4.dat",
    5.0: "scl_pmos_transfer_vsd_5.dat",
}

transfer = {}
rows = []
for vsd, filename in transfer_files.items():
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Missing transfer file: {path}")
        continue
    vsg, id_a = load_wrdata(path, x_col=1)
    gm = np.gradient(id_a, vsg)
    gm_id = np.divide(gm, id_a, out=np.full_like(gm, np.nan), where=id_a > 1e-12)
    transfer[vsd] = {"vsg": vsg, "id": id_a, "gm": gm, "gm_id": gm_id}
    for i in range(len(vsg)):
        rows.append({
            "VSD_V": vsd, "VSG_V": vsg[i], "ID_A": id_a[i],
            "ID_uA": id_a[i]*1e6, "gm_S": gm[i], "gm_uS": gm[i]*1e6,
            "gm_ID_per_V": gm_id[i]
        })

if not transfer:
    raise FileNotFoundError("No PMOS transfer files loaded. Check DATA_DIR and filenames.")

transfer_df = pd.DataFrame(rows)
transfer_df.to_csv(OUT_TABLES / "scl08_pmos_transfer_parameters.csv", index=False)
display(transfer_df.iloc[::10].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for vsd, d in transfer.items():
    ax.plot(d["vsg"], d["id"]*1e6, label=f"VSD = {vsd:g} V")
ax.set(xlabel=r"$V_{SG}$ (V)", ylabel=r"$|I_D|$ ($\mu$A)",
       title="SCL 0.8 µm PMOS Transfer Characteristics")
ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_pmos_transfer.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for vsd, d in transfer.items():
    ax.plot(d["vsg"], d["gm"]*1e6, label=f"VSD = {vsd:g} V")
ax.set(xlabel=r"$V_{SG}$ (V)", ylabel=r"$g_m$ ($\mu$S)",
       title="PMOS Transconductance")
ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_pmos_gm.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for vsd, d in transfer.items():
    ax.plot(d["vsg"], d["gm_id"], label=f"VSD = {vsd:g} V")
ax.set(xlabel=r"$V_{SG}$ (V)", ylabel=r"$g_m/|I_D|$ (V$^{-1}$)",
       title="PMOS Transconductance Efficiency")
ax.set_ylim(bottom=0); ax.grid(True); ax.legend(); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_pmos_gm_id.png", dpi=200)
plt.show()

## PMOS output characteristics

The expected input files are `scl_pmos_output_vsg_0p5.dat`, `_1.dat`, `_1p5.dat`, `_2.dat`, `_2p5.dat`, `_3.dat`, `_3p5.dat`, `_4.dat`, `_4p5.dat`, and `_5.dat`. This cell plots `|ID|` against `VSD` for each fixed `VSG`.

In [ ]:
output_files = {
    0.5: "scl_pmos_output_vsg_0p5.dat",
    1.0: "scl_pmos_output_vsg_1.dat",
    1.5: "scl_pmos_output_vsg_1p5.dat",
    2.0: "scl_pmos_output_vsg_2.dat",
    2.5: "scl_pmos_output_vsg_2p5.dat",
    3.0: "scl_pmos_output_vsg_3.dat",
    3.5: "scl_pmos_output_vsg_3p5.dat",
    4.0: "scl_pmos_output_vsg_4.dat",
    4.5: "scl_pmos_output_vsg_4p5.dat",
    5.0: "scl_pmos_output_vsg_5.dat",
}

fig, ax = plt.subplots(figsize=(9, 6))
output_rows = []
for vsg, filename in output_files.items():
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Missing output file: {path}")
        continue
    vsd, id_a = load_wrdata(path, x_col=1)
    ax.plot(vsd, id_a*1e6, label=f"VSG = {vsg:g} V")
    gds = np.gradient(id_a, vsd)
    for i in range(len(vsd)):
        output_rows.append({
            "VSG_V": vsg, "VSD_V": vsd[i], "ID_uA": id_a[i]*1e6,
            "gds_uS": gds[i]*1e6,
            "ro_kOhm": (1/gds[i]/1e3) if gds[i] > 0 else np.nan
        })

ax.set(xlabel=r"$V_{SD}$ (V)", ylabel=r"$|I_D|$ ($\mu$A)",
       title="SCL 0.8 µm PMOS Output Characteristics")
ax.grid(True); ax.legend(ncol=2); fig.tight_layout()
fig.savefig(OUT_FIGURES / "scl08_pmos_output.png", dpi=200)
plt.show()

if output_rows:
    output_df = pd.DataFrame(output_rows)
    output_df.to_csv(OUT_TABLES / "scl08_pmos_output_parameters.csv", index=False)
    display(output_df.iloc[::20].round(4))

## Interpretation notes

- `gm` and `gm/ID` are numerical estimates from the transfer sweep.
- `gds` and `ro` are estimated from the output sweep.
- This notebook does **not** combine `gm` and `gds` into a matched intrinsic-gain table automatically, because that requires confirming that the chosen transfer and output bias points are consistent.
- Very low-current points can produce misleading ratios.
- Validate model compatibility and the netlist's bias conditions before using extracted values to size the OTA.